In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_theta,R2diff_ZZx1_theta,R2_ZZx2_theta,R2diff_ZZx2_theta,...,R2_LSG_1_theta,R2diff_LSG_1_theta,R2_LSG_2_theta,R2diff_LSG_2_theta,R2_ZZx1_inv_theta,R2diff_ZZx1_inv_theta,R2_zzx2_inv2_theta,R2diff_zzx2_inv2_theta,R2_semiCirc_theta,R2diff_semiCirc_theta
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9468,[1],0.5,0.5,0.01,9468,0.812644,0.637816,-0.866878,0.383926,...,0.779333,0.583208,-6.126531,0.283605,-0.074995,0.286269,-7.858811,-0.052684,-36.864434,-0.346101
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed9583,[1],0.5,0.5,0.01,9583,0.335420,0.663848,-0.481250,0.430286,...,0.804239,0.598235,-3.419127,0.337881,-0.559936,0.406081,-8.470865,0.039391,-29.539623,-0.180141
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed2302,[1],0.5,0.5,0.01,2302,0.606371,0.635331,-0.861561,0.384327,...,0.851640,0.587906,-4.496989,0.313197,-0.196650,0.326580,-7.976547,-0.028246,-33.820807,-0.277783
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed134,[1],0.5,0.5,0.01,134,0.723993,0.643883,-0.936011,0.385927,...,0.724940,0.597476,-5.251368,0.309798,-0.112615,0.314128,-7.958574,-0.062566,-38.820195,-0.377207
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed9940,[1],0.5,0.5,0.01,9940,0.167741,0.617201,-0.056641,0.409887,...,-0.444151,0.526502,-3.264614,0.288599,-0.775709,0.352238,-7.955050,0.112877,-17.804839,0.011120
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,model_arch100_r0.9_Ld0.7_Lp0.3_seed9468,[100],0.7,0.3,0.90,9468,0.886052,0.784444,0.830051,0.516487,...,0.782084,0.603156,-2.470016,0.390286,-5.685433,0.478214,-17.852422,0.086295,-30.151924,-0.201547
2996,model_arch100_r0.9_Ld0.7_Lp0.3_seed9583,[100],0.7,0.3,0.90,9583,0.586011,0.817200,0.379857,0.494261,...,0.758888,0.613862,-3.511392,0.366532,-3.078979,0.477496,-11.388272,0.076856,-37.631057,-0.382723
2997,model_arch100_r0.9_Ld0.7_Lp0.3_seed2302,[100],0.7,0.3,0.90,2302,0.950053,0.819621,0.340812,0.493609,...,0.771804,0.629347,-3.047201,0.351130,-3.666802,0.407117,-16.814750,-0.085397,-45.537406,-0.522895
2998,model_arch100_r0.9_Ld0.7_Lp0.3_seed134,[100],0.7,0.3,0.90,134,0.677513,0.565140,0.665587,0.498296,...,0.580402,0.541552,-2.295448,0.336575,-0.315840,0.450448,-7.445994,0.165997,-22.778516,-0.064251


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "ZZx1":  "Train",
    "ZZx2":     "Val",
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZxReto":  "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.4
w_train = 0.4
w_test = 0.1

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
40,model_arch2_r0.01_Ld0.3_Lp0.7_seed9468,[2],0.627581,0.789176,-4.288857,-0.287936
224,model_arch8_r0.01_Ld0.3_Lp0.7_seed9940,[8],0.624359,0.722051,-4.339227,-0.352882
72,model_arch3_r0.01_Ld0.3_Lp0.7_seed2302,[3],0.667542,0.821541,-4.581008,-0.365994
129,model_arch5_r0.9_Ld0.5_Lp0.5_seed9940,[5],0.765713,0.847882,-4.768192,-0.370221
2080,model_arch70_r0.01_Ld0.3_Lp0.7_seed9468,[70],0.662233,0.311184,-3.598243,-0.371065
152,model_arch6_r0.01_Ld0.5_Lp0.5_seed2302,[6],0.746335,0.894363,-5.060834,-0.372477
77,model_arch3_r0.9_Ld0.3_Lp0.7_seed2302,[3],0.580134,0.636878,-4.383237,-0.376104
99,model_arch4_r0.9_Ld0.5_Lp0.5_seed9940,[4],0.659516,0.853527,-5.144299,-0.385818
708,model_arch24_r0.9_Ld0.3_Lp0.7_seed134,[24],0.594085,-0.100663,-2.718262,-0.387387
2089,model_arch70_r0.9_Ld0.3_Lp0.7_seed9940,[70],0.705284,0.506545,-3.958680,-0.394884



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZxReto_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
40,model_arch2_r0.01_Ld0.3_Lp0.7_seed9468,[2],0.627581,0.789176,-6.950085,-9.941175,-2.721907,-1.215966,-0.411615,0.091076,-8.872324,0.627581,0.789176,-4.288857,-0.287936
224,model_arch8_r0.01_Ld0.3_Lp0.7_seed9940,[8],0.624359,0.722051,-3.850089,-13.555621,-4.207556,-1.580850,-1.579931,0.606886,-6.207432,0.624359,0.722051,-4.339227,-0.352882
72,model_arch3_r0.01_Ld0.3_Lp0.7_seed2302,[3],0.667542,0.821541,-2.554526,-15.250276,-4.062741,-2.698832,-2.022823,0.697615,-6.175472,0.667542,0.821541,-4.581008,-0.365994
129,model_arch5_r0.9_Ld0.5_Lp0.5_seed9940,[5],0.765713,0.847882,-3.199217,-14.056985,-2.210048,-2.002327,-1.524377,0.728480,-11.112868,0.765713,0.847882,-4.768192,-0.370221
2080,model_arch70_r0.01_Ld0.3_Lp0.7_seed9468,[70],0.662233,0.311184,-4.929448,-9.420836,-1.096555,-1.441273,0.065832,0.382160,-8.747576,0.662233,0.311184,-3.598243,-0.371065
152,model_arch6_r0.01_Ld0.5_Lp0.5_seed2302,[6],0.746335,0.894363,-5.643274,-11.808903,-1.935394,-1.480666,-2.770109,0.800966,-12.588462,0.746335,0.894363,-5.060834,-0.372477
77,model_arch3_r0.9_Ld0.3_Lp0.7_seed2302,[3],0.580134,0.636878,-2.101101,-11.779169,-4.578037,-2.642861,-2.136614,0.553087,-7.997966,0.580134,0.636878,-4.383237,-0.376104
99,model_arch4_r0.9_Ld0.5_Lp0.5_seed9940,[4],0.659516,0.853527,-1.596710,-12.635899,-4.708190,-4.306739,-3.604092,0.661347,-9.819813,0.659516,0.853527,-5.144299,-0.385818
708,model_arch24_r0.9_Ld0.3_Lp0.7_seed134,[24],0.594085,-0.100663,-2.022046,-8.626921,-1.952273,-1.443588,0.204396,0.426425,-5.613829,0.594085,-0.100663,-2.718262,-0.387387
2089,model_arch70_r0.9_Ld0.3_Lp0.7_seed9940,[70],0.705284,0.506545,-5.089502,-9.087771,-0.269924,-1.510838,0.051668,0.576850,-12.381241,0.705284,0.506545,-3.958680,-0.394884


In [5]:
final_table.to_excel("BestModels-1l.xlsx")